# Load Data

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier

path = "/kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset"
train = pd.read_csv(f"{path}/train.csv")
test = pd.read_csv(f"{path}/test.csv")
submission = pd.read_csv(f"{path}/sample_submission.csv")

# Explore Data

In [ ]:
train.shape
test.shape

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
train["price_tier"].value_counts().sort_index()

# Prepare Data

In [ ]:
target = "price_tier"
num_cols = train.select_dtypes(include="number").columns.tolist()
num_cols.remove(target)
num_cols = [column for column in num_cols if column in test.columns]
X = train[num_cols].fillna(train[num_cols].median())
X_test = test[num_cols].fillna(train[num_cols].median())
y = train[target]

# HPT

In [ ]:
tree_rand = RandomizedSearchCV(
    DecisionTreeClassifier(random_state=42),
    {
        "max_depth": [2, 4, 6, 8, 10, 12, 16, 20, 24, 30],
        "min_samples_split": [2, 5, 10, 20, 50],
        "min_samples_leaf": [1, 2, 5, 10, 20]
    },
    n_iter=20,
    cv=3,
    random_state=42
)
tree_rand.fit(X, y)

In [ ]:
tree_depth = tree_rand.best_params_["max_depth"]
tree_split = tree_rand.best_params_["min_samples_split"]
tree_leaf = tree_rand.best_params_["min_samples_leaf"]
tree_cv = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    {
        "max_depth": [max(2, tree_depth - 2), tree_depth, tree_depth + 2],
        "min_samples_split": [tree_split, tree_split + 2],
        "min_samples_leaf": [tree_leaf, tree_leaf + 1]
    },
    cv=3
)
tree_cv.fit(X, y)

In [ ]:
forest_rand = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    {
        "n_estimators": [50, 100, 150, 200, 300, 500],
        "max_depth": [None, 4, 6, 8, 10, 12, 16, 20, 24, 30],
        "min_samples_split": [2, 5, 10, 20, 50],
        "min_samples_leaf": [1, 2, 5, 10, 20],
        "max_features": ["sqrt", "log2", None]
    },
    n_iter=25,
    cv=3,
    random_state=42,
    n_jobs=-1
)
forest_rand.fit(X, y)

In [ ]:
forest_num = forest_rand.best_params_["n_estimators"]
forest_depth = forest_rand.best_params_["max_depth"]
forest_split = forest_rand.best_params_["min_samples_split"]
forest_leaf = forest_rand.best_params_["min_samples_leaf"]
forest_feat = forest_rand.best_params_["max_features"]
forest_depths = [4, 8] if forest_depth is None else [max(2, forest_depth - 2), forest_depth, forest_depth + 2]
forest_cv = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    {
        "n_estimators": [forest_num, forest_num + 50],
        "max_depth": forest_depths,
        "min_samples_split": [forest_split, forest_split + 2],
        "min_samples_leaf": [forest_leaf, forest_leaf + 1],
        "max_features": [forest_feat]
    },
    cv=3,
    n_jobs=-1
)
forest_cv.fit(X, y)

# Train Model

In [ ]:
model = tree_cv.best_estimator_
tree_preds = model.predict(X_test)

In [ ]:
print("Tree score:", model.score(X, y))

# Train Second Model

In [ ]:
forest = forest_cv.best_estimator_
forest_preds = forest.predict(X_test)

In [ ]:
print("Forest score:", forest.score(X, y))

# Create Submission

In [ ]:
preds = np.maximum(tree_preds, forest_preds)
submission[target] = preds
submission.to_csv("/kaggle/working/submission.csv", index=False)

In [ ]:
submission